# Full fine-tuning of pretrained ViT-Tiny

This notebook fine-tunes an ImageNet-pretrained ViT-Tiny on the two classes in `data/`. The classification head is replaced for two-class prediction, while the complete pretrained backbone remains trainable—**no layers are frozen**.

It uses the project's authoritative 176/45 train/validation manifest, making it directly comparable with the other replication models. The pretrained model is downloaded the first time this notebook runs.

> The shared validation set is used for checkpoint selection and reporting. It is not an independent test set.

## 1. Install dependencies
Run once in a fresh environment, then restart the kernel if requested.

In [ ]:
%pip install -q torch torchvision timm matplotlib scikit-learn

## 2. Imports and configuration

In [ ]:
import copy
import csv
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'data').is_dir() and (parent / 'model_reproductions_7_models').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data'
SPLIT_MANIFEST = DATA_DIR / 'common_split_manifest.csv'
MODEL_NAME = 'vit_tiny_patch16_224.augreg_in21k_ft_in1k'
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0
EPOCHS = 30
WARMUP_EPOCHS = 3
PATIENCE = 7
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 5e-2

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

## 3. Load ViT-Tiny and replace its classifier
`pretrained=True` loads learned ImageNet features. `num_classes=2` creates a new two-output head. The assertion confirms full fine-tuning.

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=2)
model = model.to(device)

# Be explicit: the backbone and new classifier are both trainable.
for parameter in model.parameters():
    parameter.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert total_params == trainable_params, 'Some parameters are frozen.'

# Use the normalization associated with these pretrained weights.
pretrained_cfg = timm.data.resolve_model_data_config(model)
mean = pretrained_cfg['mean']
std = pretrained_cfg['std']
print(f'Model: {MODEL_NAME}')
print(f'Parameters: {total_params:,} (all {trainable_params:,} trainable)')
print(f'Pretrained normalization: mean={mean}, std={std}')

## 4. Dataset and shared 80/20 split
The project manifest fixes the exact 176 training and 45 validation filenames used by every replication model. The source BMPs are grayscale, but pretrained ViT-Tiny expects three channels, so their values are replicated into RGB.

In [ ]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.90, 1.00), ratio=(0.98, 1.02)),
    transforms.RandomAffine(degrees=3, translate=(0.02, 0.02), scale=(0.98, 1.02)),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

train_view = datasets.ImageFolder(DATA_DIR, transform=train_transform)
eval_view = datasets.ImageFolder(DATA_DIR, transform=eval_transform)
assert train_view.samples == eval_view.samples
class_names = train_view.classes
num_classes = len(class_names)
assert num_classes == 2, f'Expected 2 classes, found {num_classes}: {class_names}'

with SPLIT_MANIFEST.open(newline='') as handle:
    split_by_file = {row['file']: row['split'] for row in csv.DictReader(handle)}

sample_names = [Path(path).name for path, _ in train_view.samples]
assert set(sample_names) == set(split_by_file), 'Dataset and split manifest do not match.'
train_idx = [i for i, name in enumerate(sample_names) if split_by_file[name] == 'train']
val_idx = [i for i, name in enumerate(sample_names) if split_by_file[name] == 'validation']
assert len(train_idx) == 176 and len(val_idx) == 45

train_set = Subset(train_view, train_idx)
val_set = Subset(eval_view, val_idx)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=(device.type == 'cuda'))
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_set, shuffle=True, generator=generator, **loader_args)
val_loader = DataLoader(val_set, shuffle=False, **loader_args)

def class_counts(indices):
    return {class_names[c]: sum(train_view.targets[i] == c for i in indices)
            for c in range(num_classes)}

print('Classes:', train_view.class_to_idx)
print(f'Total: {len(train_view)} | train: {len(train_set)} | validation: {len(val_set)}')
print('Train:', class_counts(train_idx))
print('Validation:', class_counts(val_idx))

In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
display_mean = torch.tensor(mean).view(3, 1, 1)
display_std = torch.tensor(std).view(3, 1, 1)
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    restored = (image * display_std + display_mean).clamp(0, 1)
    ax.imshow(restored.permute(1, 2, 0))
    ax.set_title(class_names[label.item()])
    ax.axis('off')
plt.tight_layout()

## 5. Full fine-tuning setup
The pretrained backbone gets a cautious learning rate; the newly initialized classifier gets a larger one. Both parameter groups are updated on every training step.

In [ ]:
head_parameters = list(model.get_classifier().parameters())
head_ids = {id(p) for p in head_parameters}
backbone_parameters = [p for p in model.parameters() if id(p) not in head_ids]
assert backbone_parameters and head_parameters
assert all(p.requires_grad for p in backbone_parameters + head_parameters)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW([
    {'params': backbone_parameters, 'lr': BACKBONE_LR},
    {'params': head_parameters, 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)

warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.10, end_factor=1.0, total_iters=WARMUP_EPOCHS
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS), eta_min=1e-7
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS]
)
use_amp = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def run_epoch(model, loader, training):
    model.train(training)
    loss_sum = 0.0
    correct = 0
    seen = 0

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(inputs)
                loss = criterion(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

        loss_sum += loss.item() * targets.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        seen += targets.size(0)

    return loss_sum / seen, correct / seen

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_state = None
best_val_loss = float('inf')
epochs_without_improvement = 0
start = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, training=True)
    val_loss, val_acc = run_epoch(model, val_loader, training=False)
    backbone_lr = optimizer.param_groups[0]['lr']
    head_lr = optimizer.param_groups[1]['lr']
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    improved = val_loss < best_val_loss - 1e-4
    if improved:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f'Epoch {epoch:02d}/{EPOCHS} | backbone/head lr {backbone_lr:.1e}/{head_lr:.1e} | '
          f'train loss {train_loss:.4f}, acc {train_acc:.3f} | '
          f'val loss {val_loss:.4f}, acc {val_acc:.3f}' + (' *' if improved else ''))

    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping after {epoch} epochs.')
        break

model.load_state_dict(best_state)
print(f'Loaded best validation state ({best_val_loss:.4f}); elapsed {(time.time()-start)/60:.1f} min.')

## 6. Learning curves and shared validation evaluation

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_ran, history['train_loss'], label='train')
axes[0].plot(epochs_ran, history['val_loss'], label='validation')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy')
axes[1].plot(epochs_ran, history['train_acc'], label='train')
axes[1].plot(epochs_ran, history['val_acc'], label='validation')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy', ylim=(0, 1.02))
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()

In [ ]:
@torch.inference_mode()
def predict_loader(model, loader):
    model.eval()
    all_targets, all_predictions, all_probabilities = [], [], []
    for inputs, targets in loader:
        logits = model(inputs.to(device, non_blocking=True))
        probabilities = logits.softmax(dim=1).cpu()
        all_targets.extend(targets.tolist())
        all_predictions.extend(probabilities.argmax(dim=1).tolist())
        all_probabilities.extend(probabilities.tolist())
    return np.array(all_targets), np.array(all_predictions), np.array(all_probabilities)

y_true, y_pred, y_prob = predict_loader(model, val_loader)
validation_accuracy = (y_true == y_pred).mean()
print(f'Validation accuracy: {validation_accuracy:.3f} ({(y_true == y_pred).sum()}/{len(y_true)})\n')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=class_names, cmap='Blues', colorbar=False
)
plt.title('Fine-tuned ViT-Tiny validation confusion matrix')
plt.tight_layout()

In [ ]:
n_show = min(12, len(val_set))
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, subset_pos in zip(axes.flat, range(n_show)):
    image, target = val_set[subset_pos]
    pred = y_pred[subset_pos]
    confidence = y_prob[subset_pos, pred]
    restored = (image * display_std + display_mean).clamp(0, 1)
    ax.imshow(restored.permute(1, 2, 0))
    color = 'green' if pred == target else 'red'
    ax.set_title(f'true: {class_names[target]}\npred: {class_names[pred]} ({confidence:.0%})', color=color)
    ax.axis('off')
for ax in axes.flat[n_show:]:
    ax.axis('off')
plt.tight_layout()

## Reading the comparison

- The scratch and fine-tuning notebooks use the exact same manifest, so their validation results are paired fairly.
- These are validation results, not independent test results; the validation images also guide checkpoint selection.
- Fine-tuning should usually converge faster and overfit less than scratch training on 221 images.
- For paper-quality evidence, repeat both methods with several seeds and report mean ± standard deviation.